<a href="https://colab.research.google.com/github/Ishalllll/preconception-stunting-risk-screening/blob/main/notebooks/data_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
! pip install pygrowup

In [8]:
import os
import glob
import pandas as pd
import numpy as np

DATA_DIR = "."
OUT_FILE = "dataset_stunting.csv"

F_ROS = "bk_ar1.csv"     # roster rumah tangga: umur, sex, tgl lahir, ortu, pendidikan, kerja
F_US  = "bus_us.csv"     # Pengukuran fisik: tinggi, metode, berat
F_SC  = "bk_sc1.csv"     # Lokasi rumah tangga: sc05 urban/rural
F_KRK = "bk_krk.csv"     # observasi pewawancara: kondisi rumah & lingkungan
F_KR  = "b2_kr.csv"      # WASH: air, jamban, limbah, sampah
F_KM  = "b3b_km.csv"     # blok KM Buku 3B: km01a riwayat merokok

UMUR_MAKS = 59

JENJANG = {
    1: 0,  90: 0,
    2: 1,  11: 1,  72: 1,
    3: 2,  4: 2,   12: 2,  73: 2,
    5: 3,  6: 3,   15: 3,  74: 3,
    13: 4, 60: 4,  61: 4,  62: 4,  63: 4,
}

# pengelompokan WASH (WHO/JMP 2018 Core Questions)
AIR_LAYAK   = {1, 2, 5, 10}  # pipa, sumur pompa(asumsi), hujan, kemasan
AIR_RAGU    = {3, 4, 8}      # sumur timba, mata air, bak penampungan
AIR_TAK     = {6, 7}         # sungai, kolam (air permukaan)

# jamban: JMP membedakan basic / limited(berbagi) / tidak layak / terbuka
JAMBAN_BASIC   = {1}                    # sendiri + septic tank
JAMBAN_LIMITED = {3, 4}                 # komunal, umum (layak tapi berbagi)
JAMBAN_TAKLAYAK = {2}                   # sendiri tanpa penampungan aman
JAMBAN_TERBUKA = {5, 6, 7, 9, 10, 11}   # sungai, ladang, selokan, kolam, kandang, laut
# skala gabungan untuk fitur tunggal: 2=basic, 1=limited, 0=tidak layak/terbuka
JAMBAN = {**{k: 2 for k in JAMBAN_BASIC},
          **{k: 1 for k in JAMBAN_LIMITED},
          **{k: 0 for k in JAMBAN_TAKLAYAK | JAMBAN_TERBUKA}}

# limbah cair & sampah: DI LUAR tangga JMP (SDG 6.3.1 & 11.6.1). Tetap dipakai
LIMBAH_AMAN  = {1, 3, 8}  # selokan mengalir, lubang permanen, lubang
SAMPAH_AMAN  = {1, 2, 5}  # tempat sampah, dibakar, lubang


def baca(nama, wajib=True):
    p = os.path.join(DATA_DIR, nama)
    if not os.path.exists(p):
        if wajib:
            raise FileNotFoundError(f"'{p}' tidak ada. Isi folder: {os.listdir(DATA_DIR)}")
        print(f"  [-] {nama} tidak ada, dilewati")
        return None
    d = pd.read_csv(p, dtype=str)
    d.columns = [c.lower().strip() for c in d.columns]
    return d


def num(s):
    return pd.to_numeric(s, errors="coerce")


jejak = []
def catat(label, n, prev=None):
    jejak.append((label, n, prev))

In [9]:
print("=" * 74)
print("recode pendidikan & status kerja")
print("=" * 74)

ros = baca(F_ROS)
for c in ("pidlink", "hhid14"):
    ros[c] = ros[c].astype(str).str.strip()
ros["_pid"]   = num(ros["pid14"])
ros["_umur"]  = num(ros["ar09"])
ros["_sex"]   = num(ros["ar07"]) # 1 = laki-laki, 3 = perempuan
ros["_ibu"]   = num(ros["ar11"])
ros["_ayah"]  = num(ros["ar10"])
ros["_a"]     = num(ros["ar01a"])

# pendidikan
ros["_pend"] = num(ros["ar16"]).map(JENJANG)
print(f"  Pendidikan berhasil direcode : {ros['_pend'].notna().mean()*100:.1f}%")

# status kerja: 1=ya, 3=tidak.
kerja = num(ros["ar15a"])
ros["_kerja"] = kerja.map({1: 1, 3: 0})
print(f"  Status kerja terisi          : {ros['_kerja'].notna().mean()*100:.1f}%")

# umur: kode >90 biasanya 'tidak tahu'
ros["_umur"] = ros["_umur"].where(ros["_umur"] <= 110)

recode pendidikan & status kerja
  Pendidikan berhasil direcode : 90.2%
  Status kerja terisi          : 83.8%


In [13]:
print("\n" + "="*74)
print("Tanggal wawancara asli")
print("="*74)

ivw = None
for f in sorted(glob.glob(os.path.join(DATA_DIR, "*.csv"))):
    try:
        kol = [c.lower() for c in pd.read_csv(f, nrows=0).columns]
    except Exception:
        continue
    if "ivwyrbk" in kol and "ivwmthbk" in kol:
        ivw = pd.read_csv(f, dtype=str)
        ivw.columns = [c.lower().strip() for c in ivw.columns]
        print(f"  Ketemu di : {os.path.basename(f)}")
        break

if ivw is not None:
    ivw["hhid14"] = ivw["hhid14"].astype(str).str.strip()
    ivw["_iv_b"] = num(ivw["ivwmthbk"]).where(lambda s: s.between(1, 12))
    ivw["_iv_t"] = num(ivw["ivwyrbk"]).where(lambda s: s.between(2014, 2016))
    ros = ros.merge(ivw[["hhid14", "_iv_b", "_iv_t"]].drop_duplicates("hhid14"),on="hhid14", how="left")
    print(f"Rumah tangga dapat tanggal : {ros['_iv_t'].notna().mean()*100:.1f}%")
    print(f"Sebaran bulan : {ros['_iv_b'].value_counts().sort_index().to_dict()}")
else:
    print("Asumsi Desember 2014")
    print("kolom ivwyrbk sebelum memakai hasil ini.")
    ros["_iv_b"], ros["_iv_t"] = 12.0, 2014.0

ros["_iv_b"] = ros["_iv_b"].fillna(12.0)
ros["_iv_t"] = ros["_iv_t"].fillna(2014.0)


Tanggal wawancara asli
Asumsi Desember 2014
kolom ivwyrbk sebelum memakai hasil ini.


In [14]:
print("=" * 74)
print("Kohort balita")
print("=" * 74)

lhr_thn = num(ros["ar08yr"]).where(lambda s: s.between(1991, 2019))
lhr_b = num(ros["ar08mth"]).where(lambda s: s.between(1, 12))
ros["_umur_bln"] = (ros["_iv_t"] - lhr_thn) * 12 + (ros["_iv_b"] - lhr_b)
ros["_umur_bln"] = ros["_umur_bln"].where(ros["_umur_bln"].between(0, 250))

anak = ros[ros["_a"].isin([1, 5]) & ros["_umur_bln"].between(0, UMUR_MAKS)].copy()
anak = anak.drop_duplicates("pidlink")
catat(f"Balita 0-{UMUR_MAKS} bulan", len(anak))
print(f"Balita 0-{UMUR_MAKS} bulan : {len(anak):,}")
print(f"Pembanding umur tahunan 0-4 : {(ros['_umur'].between(0,4) & ros['_a'].isin([1,5])).sum():,}")

Kohort balita
Balita 0-59 bulan : 5,962
Pembanding umur tahunan 0-4 : 6,101


In [15]:
print("=" * 74)
print("Antropometri anak + koreksi metode ukur")
print("=" * 74)

us = baca(F_US)
us["pidlink"] = us["pidlink"].astype(str).str.strip()
us["_tinggi"] = num(us["us04"])
us["_metode"] = num(us["us05"]) # 1 = berdiri, 3 = berbaring
us["_berat"]  = num(us["us06"])
us_kecil = us[["pidlink", "_tinggi", "_metode", "_berat"]].drop_duplicates("pidlink")

# batas masuk akal untuk anak
us_anak = us_kecil.copy()
us_anak["_tinggi"] = us_anak["_tinggi"].where(us_anak["_tinggi"].between(30, 150))
us_anak["_berat"]  = us_anak["_berat"].where(us_anak["_berat"].between(1, 60))

n0 = len(anak)
anak = anak.merge(us_anak, on="pidlink", how="left")
anak = anak[anak["_tinggi"].notna()].copy()
catat("+ tinggi anak valid", len(anak), n0)
print(f"  Punya tinggi valid : {len(anak):,} ({len(anak)/n0*100:.1f}%)")

# WHO: <24 bln diukur berbaring, >=24 bln berdiri. Kalau tidak sesuai, koreksi 0,7 cm.
sa = (anak["_umur_bln"] < 24) & (anak["_metode"] == 1)
sb = (anak["_umur_bln"] >= 24) & (anak["_metode"] == 3)
print(f"<24 bln diukur berdiri    : {sa.sum():,}")
print(f">=24 bln diukur berbaring : {sb.sum():,}")
anak["_tinggi_k"] = anak["_tinggi"] + 0.7 * sa - 0.7 * sb

Antropometri anak + koreksi metode ukur
  Punya tinggi valid : 5,207 (87.3%)
<24 bln diukur berdiri    : 405
>=24 bln diukur berbaring : 94


In [16]:
print("=" * 74)
print("TAHAP 5 : HAZ & label stunting")
print("=" * 74)

from pygrowup import Calculator
calc = Calculator(adjust_height_data=False, adjust_weight_scores=False, include_cdc=False, logger_name="pg", log_level="CRITICAL")

def haz(b):
    try:
        return float(calc.lhfa(b["_tinggi_k"], b["_umur_bln"],"M" if b["_sex"] == 1 else "F"))
    except Exception:
        return np.nan

anak["haz"] = anak.apply(haz, axis=1)
n0 = len(anak)
anak = anak[anak["haz"].between(-6, 6)].copy()      # WHO: di luar ini tidak masuk akal
catat("+ HAZ masuk akal (-6..6)", len(anak), n0)
print(f"  Dibuang karena HAZ mustahil : {n0 - len(anak):,}")

anak["stunting"] = (anak["haz"] < -2).astype(int)
print(f"Sampel : {len(anak):,}   Stunting : {anak['stunting'].sum():,} ({anak['stunting'].mean()*100:.2f}%)")

TAHAP 5 : HAZ & label stunting
  Dibuang karena HAZ mustahil : 92
Sampel : 5,115   Stunting : 1,461 (28.56%)


In [17]:
print("=" * 74)
print("Fitur ibu & ayah")
print("=" * 74)

cari = ros[["hhid14", "_pid", "pidlink", "_umur", "_pend", "_kerja"]] \
        .drop_duplicates(["hhid14", "_pid"])

ibu = cari.rename(columns={"_pid": "_ibu", "pidlink": "pid_ibu", "_umur": "ibu_umur", "_pend": "ibu_pendidikan", "_kerja": "ibu_bekerja"})
anak = anak.merge(ibu, on=["hhid14", "_ibu"], how="left").drop_duplicates("pidlink")

ayah = cari.rename(columns={"_pid": "_ayah", "pidlink": "pid_ayah", "_umur": "ayah_umur", "_pend": "ayah_pendidikan", "_kerja": "ayah_bekerja"})
anak = anak.merge(ayah, on=["hhid14", "_ayah"], how="left").drop_duplicates("pidlink")

print(f"Ibu  : {anak['pid_ibu'].notna().mean()*100:.1f}%")
print(f"Ayah : {anak['pid_ayah'].notna().mean()*100:.1f}%")

anak["ibu_usia_konsepsi"] = (anak["ibu_umur"] * 12 - anak["_umur_bln"] - 9) / 12.0
anak["ibu_usia_konsepsi"] = anak["ibu_usia_konsepsi"].where(anak["ibu_usia_konsepsi"].between(10, 60))
print(f"Usia ibu saat konsepsi terisi : {anak['ibu_usia_konsepsi'].notna().mean()*100:.1f}%")

# tinggi badan ibu
tinggi_ibu = us_kecil[["pidlink", "_tinggi"]].rename(columns={"pidlink": "pid_ibu", "_tinggi": "ibu_tinggi"})
tinggi_ibu["ibu_tinggi"] = tinggi_ibu["ibu_tinggi"].where(tinggi_ibu["ibu_tinggi"].between(120, 200))
anak = anak.merge(tinggi_ibu, on="pid_ibu", how="left")
print(f"Tinggi ibu terisi : {anak['ibu_tinggi'].notna().mean()*100:.1f}%  <-- CEK INI")

# Tinggi badan ayah
tinggi_ayah = us_kecil[["pidlink", "_tinggi"]].rename(columns={"pidlink": "pid_ayah", "_tinggi": "ayah_tinggi"})
tinggi_ayah["ayah_tinggi"] = tinggi_ayah["ayah_tinggi"].where(tinggi_ayah["ayah_tinggi"].between(120, 220))
anak = anak.merge(tinggi_ayah, on="pid_ayah", how="left")
print(f"Tinggi ayah terisi : {anak['ayah_tinggi'].notna().mean()*100:.1f}%")

Fitur ibu & ayah
Ibu  : 98.6%
Ayah : 90.2%
Usia ibu saat konsepsi terisi : 98.5%
Tinggi ibu terisi : 96.1%  <-- CEK INI
Tinggi ayah terisi : 77.8%


In [18]:
print("=" * 74)
print("Desa/kota")
print("=" * 74)

sc = baca(F_SC, wajib=False)
if sc is not None:
    sc["hhid14"] = sc["hhid14"].astype(str).str.strip()
    sc["urban"] = num(sc["sc05"]).map({1: 1, 2: 0})       # 1=urban, 2=rural
    anak = anak.merge(sc[["hhid14", "urban"]].drop_duplicates("hhid14"),on="hhid14", how="left")
    print(f"Terisi : {anak['urban'].notna().mean()*100:.1f}%   urban {anak['urban'].mean()*100:.1f}%")

Desa/kota
Terisi : 100.0%   urban 57.8%


In [19]:
print("=" * 74)
print("Kondisi rumah & lingkungan (observasi)")
print("=" * 74)

krk = baca(F_KRK, wajib=False)
KRK = {
    "krk02a": "limbah_dekat_rumah",
    "krk02b": "tumpukan_sampah",
    "krk02c": "air_tergenang",
    "krk02d": "dekat_kandang",
    "krk02e": "ventilasi_cukup",
    "krk02i": "dapur_kamar_menyatu",
    "krk05a": "luas_rumah_m2",
    "krk06": "jumlah_kamar",
    "krk08": "bahan_lantai",
    "krk09": "bahan_dinding",
    "krk10": "bahan_atap",
}
if krk is not None:
    krk["hhid14"] = krk["hhid14"].astype(str).str.strip()
    pk = {k: v for k, v in KRK.items() if k in krk.columns}
    sub = krk[["hhid14"] + list(pk)].rename(columns=pk).drop_duplicates("hhid14")
    for c in pk.values():
        sub[c] = num(sub[c])
    anak = anak.merge(sub, on="hhid14", how="left")
    print(f"Kolom dipakai : {len(pk)} dari {len(KRK)}")
    print(f"Terisi : {anak['bahan_lantai'].notna().mean()*100:.1f}%")

Kondisi rumah & lingkungan (observasi)
Kolom dipakai : 11 dari 11
Terisi : 100.0%


In [21]:
print("=" * 74)
print("WASH (air, jamban, limbah, sampah)")
print("=" * 74)

kr = baca(F_KR, wajib=False)
if kr is not None:
    kr["hhid14"] = kr["hhid14"].astype(str).str.strip()
    w = kr[["hhid14"]].copy()
    w["air_minum_kode"] = num(kr["kr13"])

    kr16 = num(kr["kr16"]) if "kr16" in kr.columns else pd.Series(np.nan, index=kr.index)
    kr17 = num(kr["kr17"])
    w["air_mck_kode"] = np.where(kr16 == 1, w["air_minum_kode"], kr17)
    w["air_mck_sama"] = (kr16 == 1).astype(float).where(kr16.notna())
    w["jamban_kode"] = num(kr["kr20"])
    w["limbah_kode"] = num(kr["kr21"])
    w["sampah_kode"] = num(kr["kr22"])
    w["listrik"] = num(kr["kr11"]).map({1: 1, 3: 0})

    def gol_air(s):
        return np.select([s.isin(AIR_LAYAK), s.isin(AIR_RAGU), s.isin(AIR_TAK)], [2, 1, 0], default=np.nan)
    w["air_minum_layak"] = gol_air(w["air_minum_kode"])
    w["air_mck_layak"] = gol_air(w["air_mck_kode"])
    w["jamban_layak"] = w["jamban_kode"].map(JAMBAN)
    w["limbah_aman"] = w["limbah_kode"].isin(LIMBAH_AMAN).astype(float).where(w["limbah_kode"].notna())
    w["sampah_aman"] = w["sampah_kode"].isin(SAMPAH_AMAN).astype(float).where(w["sampah_kode"].notna())

    anak = anak.merge(w.drop_duplicates("hhid14"), on="hhid14", how="left")
    print(f"  Terisi (jamban)  : {anak['jamban_kode'].notna().mean()*100:.1f}%")
    print(f"  Terisi (air MCK) : {anak['air_mck_kode'].notna().mean()*100:.1f}%")
    print(f"  Air minum kemasan (kode 10) : {(anak['air_minum_kode']==10).mean()*100:.1f}%")
    if "air_mck_sama" in anak.columns:
        print(f"  Air MCK = air minum (kr16=Ya)  : {anak['air_mck_sama'].mean()*100:.1f}%")
    print(f"  Buang air besar terbuka     : {(anak['jamban_layak']==0).mean()*100:.1f}%")

    # Uji asumsi berapa rumah yang berubah sejak 2007
    reno = {"kr24b7": "pipa air", "kr24b8": "saluran limbah","kr24b1": "bangun rumah baru", "kr24b3": "ganti atap"}
    ada_reno = {k: v for k, v in reno.items() if k in kr.columns}
    if ada_reno:
        print("\n Asumsi renovasi sejak 2007")
        r = kr[["hhid14"] + list(ada_reno)].drop_duplicates("hhid14")
        r["hhid14"] = r["hhid14"].astype(str).str.strip()
        rr = anak[["hhid14"]].merge(r, on="hhid14", how="left")
        for k, v in ada_reno.items():
            p = (num(rr[k]) == 1).mean() * 100
            print(f"{v:20s} {p:5.1f}% rumah")

WASH (air, jamban, limbah, sampah)
  Terisi (jamban)  : 100.0%
  Terisi (air MCK) : 100.0%
  Air minum kemasan (kode 10) : 37.8%
  Air MCK = air minum (kr16=Ya)  : 54.4%
  Buang air besar terbuka     : 15.8%

 Asumsi renovasi sejak 2007
pipa air               0.7% rumah
saluran limbah         0.5% rumah
bangun rumah baru      1.0% rumah
ganti atap             1.1% rumah


In [22]:
print("=" * 74)
print("Kebiasaan Merokok")
print("=" * 74)

km = baca(F_KM)
km_kecil = km[["pidlink", "km01a"]].copy()
km_kecil["ayah_merokok"] = num(km_kecil["km01a"]).map({1: 1, 3: 0})
km_kecil = (km_kecil.dropna(subset=["pidlink"]).drop_duplicates("pidlink").rename(columns={"pidlink": "pid_ayah"}))

anak = anak.drop(columns=["ayah_merokok"], errors="ignore")
anak = anak.merge(km_kecil[["pid_ayah", "ayah_merokok"]],on="pid_ayah", how="left")

print(f"  Merokok ayah terisi : {anak['ayah_merokok'].notna().mean()*100:.1f}%")
if anak["ayah_merokok"].notna().any() and "stunting" in anak.columns:
    print("\nprevalensi stunting per status merokok ayah:")
    print((anak.groupby("ayah_merokok")["stunting"].mean()*100).round(2).to_string())

Kebiasaan Merokok
  Merokok ayah terisi : 83.5%

prevalensi stunting per status merokok ayah:
ayah_merokok
0.0    26.25
1.0    28.56


In [25]:
print("=" * 74)
print("Tabel Final")
print("=" * 74)

FITUR = [
    # Kondisi tetap sebelum anak lahir, tidak bisa diintervensi
    "ibu_tinggi", "ibu_pendidikan", "ayah_pendidikan", "ibu_usia_konsepsi","ayah_tinggi",
    # struktural
    "urban", "ibu_bekerja", "ayah_bekerja","ayah_merokok",
    # kontrol anak
    "_umur_bln", "_sex",
    # kondisi rumah
    "limbah_dekat_rumah", "tumpukan_sampah", "air_tergenang", "dekat_kandang",
    "ventilasi_cukup", "dapur_kamar_menyatu", "luas_rumah_m2", "jumlah_kamar",
    "bahan_lantai", "bahan_dinding", "bahan_atap",
    # WASH
    "air_minum_layak", "air_mck_layak", "air_mck_sama", "jamban_layak",
    "limbah_aman", "sampah_aman", "listrik",
]
FITUR = [c for c in FITUR if c in anak.columns]
KUNCI = ["pidlink", "hhid14", "pid_ibu", "pid_ayah"]
AUDIT = ["haz", "_tinggi", "_tinggi_k", "_berat", "_metode", "ibu_umur",
         "air_minum_kode", "air_mck_kode", "jamban_kode", "limbah_kode", "sampah_kode"]
AUDIT = [c for c in AUDIT if c in anak.columns]

final = anak[[c for c in KUNCI if c in anak.columns] + FITUR + ["stunting"] + AUDIT].copy()
final = final.rename(columns={"_umur_bln": "umur_bulan", "_sex": "jenis_kelamin"})
FITUR = ["umur_bulan" if c == "_umur_bln" else "jenis_kelamin" if c == "_sex" else c for c in FITUR]

print(f"Baris : {len(final):,}   Fitur : {len(FITUR)}")
print("\nKelengkapan tiap fitur:")
for c in FITUR:
    p = final[c].notna().mean() * 100
    tanda = " (rendah)" if p < 85 else ""
    print(f"    {c:24s} {p:5.1f}%{tanda}")

final.to_csv(os.path.join(DATA_DIR, OUT_FILE), index=False)

Tabel Final
Baris : 5,115   Fitur : 29

Kelengkapan tiap fitur:
    ibu_tinggi                96.1%
    ibu_pendidikan            98.1%
    ayah_pendidikan           89.5%
    ibu_usia_konsepsi         98.5%
    ayah_tinggi               77.8% (rendah)
    urban                    100.0%
    ibu_bekerja               98.4%
    ayah_bekerja              89.7%
    ayah_merokok              83.5% (rendah)
    umur_bulan               100.0%
    jenis_kelamin            100.0%
    limbah_dekat_rumah       100.0%
    tumpukan_sampah          100.0%
    air_tergenang            100.0%
    dekat_kandang            100.0%
    ventilasi_cukup          100.0%
    dapur_kamar_menyatu      100.0%
    luas_rumah_m2              0.3% (rendah)
    jumlah_kamar             100.0%
    bahan_lantai             100.0%
    bahan_dinding            100.0%
    bahan_atap               100.0%
    air_minum_layak           98.6%
    air_mck_layak             98.5%
    air_mck_sama             100.0%
    jamba

In [24]:
# cek ubang air kemasan
kr = pd.read_csv("b2_kr.csv", dtype=str, low_memory=False)
for c in ["kr13", "kr16", "kr17"]:
    kr[c] = pd.to_numeric(kr[c], errors="coerce")

tab = pd.crosstab(kr["kr13"] == 10, kr["kr16"], dropna=False)
tab.index = ["kr13 bukan kemasan", "kr13 = 10 (kemasan)"]
print("=== kr13 kemasan x kr16 (1=sumber sama, 3=beda) ===")
print(tab.to_string())

n_lubang = ((kr["kr13"] == 10) & (kr["kr16"] == 1)).sum()
print(f"\nrumah tangga yang kena lubang: {n_lubang} ({n_lubang/len(kr)*100:.1f}% dari {len(kr)} rumah tangga)")

# Sanity check: kalau kr16 = 1, kr17 harusnya kosong semua (based on pemahaman aturan kuesioner)
print("\nkr17 terisi padahal kr16 = 1 (harusnya 0):", kr.loc[kr["kr16"] == 1, "kr17"].notna().sum())

# Seberapa kembar dua kolom air
d = pd.read_csv("dataset_stunting.csv", low_memory=False)
if {"air_minum_layak", "air_mck_layak"} <= set(d.columns):
    sama = (d["air_minum_layak"] == d["air_mck_layak"]).mean() * 100
    print(f"\nair_minum_layak == air_mck_layak: {sama:.1f}% baris")

=== kr13 kemasan x kr16 (1=sumber sama, 3=beda) ===
kr16                  1.0   3.0
kr13 bukan kemasan   8277  1277
kr13 = 10 (kemasan)    92  5539

rumah tangga yang kena lubang: 92 (0.6% dari 15185 rumah tangga)

kr17 terisi padahal kr16 = 1 (harusnya 0): 0

air_minum_layak == air_mck_layak: 89.0% baris
